In [2]:
# ========================================
# 1 Google Drive マウント
# ========================================
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted!")

Mounted at /content/drive
✅ Google Drive mounted!


In [3]:
## ============================================================
## 【セル2】SSH + Keep-alive + 自動バックアップ セットアップ
## ============================================================
## cloudflared は watchdog で監視され、落ちたら自動再起動します

import glob
import os
import re
import subprocess
import threading
import time
from datetime import datetime
from IPython.display import display, Javascript

# ==================== 設定 ====================
PUBLIC_KEY = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJPiuYfkrq+lqJGxN4XLCE2Nia5nyWxuCSnrbvHBUBzL solafune-colab"
WORK_DIR = '/content/workspace/solafune_construction-cost-prediction'
BACKUP_DIR = f'{WORK_DIR}/.colab_local/backups'
AUTO_BACKUP_INTERVAL = 300  # 5分（秒）
TUNNEL_LOG_PATH = '/tmp/cloudflared.log'
TUNNEL_URL_PATH = '/tmp/colab_ssh_url.txt'
WATCHDOG_INTERVAL = 15
# ==============================================


def ensure_sshd_options(path, options):
    with open(path, 'r', encoding='utf-8') as f:
        conf = f.read()
    append = []
    for line in options:
        key = line.split()[0]
        if re.search(rf'(?m)^\s*{re.escape(key)}\b', conf):
            continue
        append.append(line)
    if append:
        with open(path, 'a', encoding='utf-8') as f:
            f.write('\n' + '\n'.join(append) + '\n')


def extract_url_from_log():
    if not os.path.exists(TUNNEL_LOG_PATH):
        return None
    with open(TUNNEL_LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        log = f.read()
    m = re.search(r'https://([a-z0-9\-]+\.trycloudflare\.com)', log)
    return m.group(1) if m else None


def persist_tunnel_url(url):
    with open(TUNNEL_URL_PATH, 'w', encoding='utf-8') as f:
        f.write(url + '\n')


print("📦 Installing openssh-server...")
subprocess.run(['apt-get', 'install', '-qq', '-o=Dpkg::Use-Pty=0', 'openssh-server'], check=True, capture_output=True)

print("🔑 Setting up public key authentication...")
os.makedirs('/root/.ssh', exist_ok=True)
with open('/root/.ssh/authorized_keys', 'w') as f:
    f.write(PUBLIC_KEY + '\n')
os.chmod('/root/.ssh/authorized_keys', 0o600)

print("⚙️ Configuring SSH daemon...")
ensure_sshd_options('/etc/ssh/sshd_config', [
    'PermitRootLogin yes',
    'PubkeyAuthentication yes',
    'ClientAliveInterval 60',
    'ClientAliveCountMax 10',
    'TCPKeepAlive yes',
])

print("🚀 Restarting SSH service...")
subprocess.run(['service', 'ssh', 'restart'], check=True, capture_output=True)

print("☁️ Installing cloudflared...")
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], check=True)
subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], check=True, capture_output=True)

for d in ['', '/data', '/notebooks', '/src', '/models', '/submissions']:
    os.makedirs(f'{WORK_DIR}{d}', exist_ok=True)
os.makedirs(BACKUP_DIR, exist_ok=True)
print(f"✅ Workspace: {WORK_DIR}")


def keepalive():
    while True:
        _ = sum(range(1000))
        time.sleep(30)
threading.Thread(target=keepalive, daemon=True).start()
print("✅ Keep-alive (Python) started")

display(Javascript('''
(function() {
    function ping() {
        console.log("🔄 keepalive " + new Date().toLocaleTimeString());
        const toolbar = document.querySelector("#toolbar");
        if (toolbar) toolbar.click();
    }
    if (window.colabKeepAliveInterval) clearInterval(window.colabKeepAliveInterval);
    window.colabKeepAliveInterval = setInterval(ping, 45000);
    ping();
})();
'''))
print("✅ Keep-alive (JavaScript) started - 45sec interval")


def auto_backup():
    backup_file = f'{BACKUP_DIR}/workspace_latest.tar.gz'
    while True:
        time.sleep(AUTO_BACKUP_INTERVAL)
        try:
            if os.path.exists(WORK_DIR) and os.listdir(WORK_DIR):
                ts = datetime.now().strftime('%Y%m%d_%H%M%S')
                subprocess.run([
                    'tar', '-czf', backup_file,
                    '--exclude=.git',
                    '--exclude=.colab_local',
                    '-C', WORK_DIR, '.'
                ], check=True, capture_output=True)

                hour_mark = datetime.now().strftime('%Y%m%d_%H')
                hourly_backup = f'{BACKUP_DIR}/workspace_{hour_mark}00.tar.gz'
                if not os.path.exists(hourly_backup):
                    subprocess.run(['cp', backup_file, hourly_backup], check=True)

                backups = sorted(glob.glob(f'{BACKUP_DIR}/workspace_*.tar.gz'))
                for old in backups[:-10]:
                    os.remove(old)

                print(f"🔄 Auto-backup: {ts}")
        except Exception as e:
            print(f"⚠️ Auto-backup failed: {e}")

threading.Thread(target=auto_backup, daemon=True).start()
print(f"✅ Auto-backup started (every {AUTO_BACKUP_INTERVAL//60} min)")

state = {'proc': None, 'url': None, 'log': None}


def launch_tunnel():
    if state['log'] is not None and not state['log'].closed:
        state['log'].close()
    state['log'] = open(TUNNEL_LOG_PATH, 'ab', buffering=0)
    state['proc'] = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22'],
        stdout=state['log'],
        stderr=subprocess.STDOUT,
    )
    print(f"🔗 cloudflared started (pid={state['proc'].pid})")


def tunnel_watchdog():
    while True:
        proc = state['proc']
        needs_restart = proc is None or proc.poll() is not None
        if needs_restart:
            if proc is not None:
                print(f"⚠️ cloudflared exited (code={proc.returncode}); restarting...")
            launch_tunnel()
            time.sleep(4)

        url = extract_url_from_log()
        if url and url != state['url']:
            state['url'] = url
            persist_tunnel_url(url)
            print("=" * 60)
            print(f"📍 URL: {url}")
            print(f"🔧 WSLで実行: update-colab {url}")
            print("🔌 ssh -i ~/.ssh/solafune_colab -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ProxyCommand=\"cloudflared access ssh --hostname %h\" root@" + url)
            print("=" * 60)

        sshd_ok = subprocess.run(['pgrep', '-x', 'sshd'], capture_output=True).returncode == 0
        if not sshd_ok:
            print("⚠️ sshd not found; restarting ssh service...")
            subprocess.run(['service', 'ssh', 'restart'], capture_output=True)

        time.sleep(WATCHDOG_INTERVAL)

threading.Thread(target=tunnel_watchdog, daemon=True).start()
print("✅ cloudflared watchdog started")

for _ in range(8):
    time.sleep(1)
    current = extract_url_from_log()
    if current:
        break

print("\n✅ このセルは完了しました。他のセルを実行できます！")
print("💡 URLは /tmp/colab_ssh_url.txt にも保存されます")


📦 Installing openssh-server...
🔑 Setting up public key authentication...
⚙️ Configuring SSH daemon...
🚀 Restarting SSH service...
☁️ Installing cloudflared...
✅ Workspace: /content/workspace/solafune_construction-cost-prediction
✅ Keep-alive (Python) started


<IPython.core.display.Javascript object>

✅ Keep-alive (JavaScript) started - 45sec interval
✅ Auto-backup started (every 5 min)
✅ cloudflared watchdog started
🔗 cloudflared started (pid=3286)

✅ このセルは完了しました。他のセルを実行できます！
💡 URLは /tmp/colab_ssh_url.txt にも保存されます
📍 URL: then-enquiries-tech-humans.trycloudflare.com
🔧 WSLで実行: update-colab then-enquiries-tech-humans.trycloudflare.com
🔌 ssh -i ~/.ssh/solafune_colab -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ProxyCommand="cloudflared access ssh --hostname %h" root@then-enquiries-tech-humans.trycloudflare.com


In [ ]:
## ============================================================
## 3 opencode / codex セットアップ
## ============================================================
%%bash
set -euo pipefail

apt-get update -o Acquire::Languages=none -y
apt-get install -y ca-certificates curl gnupg git

curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get install -y nodejs

node -v
npm -v

npm i -g @openai/codex

# OpenCode install
curl -fsSL https://opencode.ai/install | bash
export PATH="$HOME/.opencode/bin:$PATH"

BASHRC="$HOME/.bashrc"
if grep -q "BEGIN OPENCODE CONFIG" "$BASHRC"; then
  sed -i '/# BEGIN OPENCODE CONFIG/,/# END OPENCODE CONFIG/d' "$BASHRC"
fi

cat <<'EOF' >> "$BASHRC"

# BEGIN OPENCODE CONFIG
export PATH="$HOME/.opencode/bin:$PATH"
export CODEX_HOME="/content/drive/MyDrive/.codex"

alias oc='opencode'
# END OPENCODE CONFIG
EOF

bash -lc 'source ~/.bashrc'
opencode --version || true
codex --version || true


In [ ]:
## ============================================================
## 4 コンペ用 GitHub リポジトリ同期
## ============================================================
## 公開 HTTPS のコンペ用 repo を指定してください（GitHub Template から事前作成済み想定）
%%bash
set -euo pipefail

REPO_URL="https://github.com/<your-account>/<your-competition-repo>.git"
WORK_DIR="/content/workspace/solafune_construction-cost-prediction"
PARENT_DIR="$(dirname "$WORK_DIR")"

if [[ "$REPO_URL" == *"<your-account>"* ]] || [[ "$REPO_URL" == *"<your-competition-repo>"* ]]; then
  echo "ERROR: REPO_URL を実際のコンペ用 GitHub リポジトリ URL に変更してください"
  exit 1
fi

mkdir -p "$PARENT_DIR"

if [ -d "$WORK_DIR/.git" ]; then
  echo "🔄 Syncing existing repo: $WORK_DIR"
  git -C "$WORK_DIR" fetch --all --prune
  git -C "$WORK_DIR" reset --hard origin/main
elif [ -d "$WORK_DIR" ]; then
  if [ -d "$WORK_DIR/.colab_local" ]; then
    echo "ERROR: $WORK_DIR exists but is not a git repo, and .colab_local backup data is present."
    echo "       自動削除を中止しました。バックアップ退避後に再実行してください。"
    exit 1
  fi
  echo "⚠️ $WORK_DIR exists but is not a git repo. Recreating..."
  rm -rf "$WORK_DIR"
  git clone "$REPO_URL" "$WORK_DIR"
else
  echo "📥 Cloning repo to $WORK_DIR"
  git clone "$REPO_URL" "$WORK_DIR"
fi

cd "$WORK_DIR"
mkdir -p .git/info
grep -qxF '.colab_local/' .git/info/exclude || echo '.colab_local/' >> .git/info/exclude
printf "\n[Repo Status]\n"
pwd
git rev-parse --short HEAD
git status --short || true


In [ ]:
## ============================================================
## 5 ワークスペース リストア（オプション）
## ============================================================
## Git管理外の作業退避（tarバックアップ）を戻すときに実行
import os
import subprocess
WORK_DIR = '/content/workspace/solafune_construction-cost-prediction'
BACKUP_FILE = f'{WORK_DIR}/.colab_local/backups/workspace_latest.tar.gz'
if os.path.exists(BACKUP_FILE):
    print("📥 Restoring workspace...")
    os.makedirs(WORK_DIR, exist_ok=True)
    subprocess.run(['tar', '-xzf', BACKUP_FILE, '-C', WORK_DIR], check=True)
    print("✅ Restored!")
else:
    print("⚠️ No backup found")


In [ ]:
## ============================================================
## 6 バックアップ（作業中・終了前に実行）
## ============================================================
## ⚠️ Git管理外の作業退避用（未commit変更の保険）
import os
import subprocess
from datetime import datetime
import glob
WORK_DIR = '/content/workspace/solafune_construction-cost-prediction'
BACKUP_DIR = f'{WORK_DIR}/.colab_local/backups'
BACKUP_FILE = f'{BACKUP_DIR}/workspace_latest.tar.gz'
os.makedirs(BACKUP_DIR, exist_ok=True)
print("📤 Backing up...")
subprocess.run(['tar', '-czf', BACKUP_FILE, '--exclude=.git', '--exclude=.colab_local', '-C', WORK_DIR, '.'], check=True)
# タイムスタンプ付きコピー
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
subprocess.run(['cp', BACKUP_FILE, f'{BACKUP_DIR}/workspace_{ts}.tar.gz'], check=True)
# 古いバックアップ削除（最新5つ保持）
for old in sorted(glob.glob(f'{BACKUP_DIR}/workspace_*.tar.gz'))[:-5]:
    os.remove(old)
print(f"✅ Backup complete! ({ts})")


In [ ]:
## ============================================================
## 7 トンネル確認・再起動・接続情報表示
## ============================================================
## 必要に応じて RESTART_TUNNEL = True にして実行
import os
import re
import time
import subprocess

RESTART_TUNNEL = False
LOG_PATH = '/tmp/cloudflared.log'
URL_PATH = '/tmp/colab_ssh_url.txt'

if RESTART_TUNNEL:
    subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
    time.sleep(2)
    print("🔄 Restarting cloudflared tunnel...")
    subprocess.Popen(
        'cloudflared tunnel --url ssh://localhost:22 > /tmp/cloudflared.log 2>&1 &',
        shell=True
    )
    time.sleep(5)

url = None
if os.path.exists(URL_PATH):
    with open(URL_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        line = f.readline().strip()
        if line:
            url = line

if not url and os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
        log = f.read()
    match = re.search(r'https://([a-z0-9\-]+\.trycloudflare\.com)', log)
    if match:
        url = match.group(1)

if url:
    print("=" * 60)
    print(f"📍 URL: {url}")
    print(f"\n🔧 WSLで実行: update-colab {url}")
    print(f"\n🔌 Terminal: ssh -i ~/.ssh/solafune_colab -o ServerAliveInterval=30 -o ServerAliveCountMax=3 -o ProxyCommand=\"cloudflared access ssh --hostname %h\" root@{url}")
    print(f"\n🔌 VS Code (Remote SSH): Host colab\n  HostName {url}\n  User root\n  ServerAliveInterval 30\n  ServerAliveCountMax 3\n  ProxyCommand cloudflared access ssh --hostname %h")
    print("=" * 60)
else:
    print("⚠️ URLが見つかりません。セル2を再実行してください。")
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH, 'r', encoding='utf-8', errors='ignore') as f:
            log = f.read()
        print("\nログ末尾:")
        print(log[-700:])

print("\n[Process Status]")
subprocess.run(['bash', '-lc', 'pgrep -a cloudflared || true'])
subprocess.run(['bash', '-lc', 'pgrep -a sshd || true'])

print("\n[Tool Versions]")
subprocess.run(['bash', '-lc', 'opencode --version || true'])
subprocess.run(['bash', '-lc', 'codex --version || true'])
